# Final evaluation and prediction output

**Final configuration**:

- main model: `LGBMRegressor(objective="quantile", alpha=0.5, learning_rate=0.03, n_estimators=300)`
- `employees` two-part model: `p_change` × `pred_size`
- base features, no relative-position columns

**Which model is deployed**
The final model is fitted on train, not refitted on train plus test. 

The reference distribution, tail thresholds and block weights downstream are all derived from train's out-of-fold predictions. A refit would require regenerating all three with no remaining data to validate them.

In [7]:
# Environment and data

from pathlib import Path

import sys
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMRegressor, LGBMClassifier

folder_01 = Path.cwd().parent / "01 EDA + Data PreProcessing"
sys.path.append(str(folder_01))

import data_prep as dp

OUT_DIR = Path("final_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

KEY = ["CompanyNumber_norm", "period_t", "period_t_plus_1"]
DATE_COLS = ["period_t", "period_t_plus_1", "available_date_t", "available_date_t_plus_1"]
META_COLS = ["primary_sector", "acct_cat_raw", "acct_cat_model"]

N_SPLITS = 5
RANDOM_STATE = 0
N_PAIRS = 2_000_000

FINAL_PARAMS = dict(learning_rate=0.03, n_estimators=300, verbose=-1, random_state=RANDOM_STATE)
TWO_PART_FIELD = "employees"

In [8]:
# Load

pairs = pd.read_csv("../01 EDA + Data PreProcessing/04_Five_CSV/03_financial_change_labels.csv",
                    dtype={"CompanyNumber_norm": str}, low_memory=False)
for c in DATE_COLS:
    pairs[c] = pd.to_datetime(pairs[c], errors="coerce")

meta = pd.read_csv("../01 EDA + Data PreProcessing/01_CompaniesSelected/UKcompanies_active_account_category_sample_100k.csv",
                   dtype={"CompanyNumber": str}, low_memory=False)
meta["CompanyNumber_norm"] = meta["CompanyNumber"].map(dp.normalise_company_number)
meta["IncorporationDate"] = pd.to_datetime(meta["IncorporationDate"], errors="coerce")

raw = pairs.merge(
    meta[["CompanyNumber_norm", "IncorporationDate", "CompanyCategory", "multi_sic_company"]],
    on="CompanyNumber_norm", how="left", validate="many_to_one")

df, _ = dp.clean_global(raw)
assignment = pd.read_csv("../01 EDA + Data PreProcessing/eda_output/split_assignment.csv",
                         dtype={"CompanyNumber_norm": str},
                         parse_dates=["period_t", "period_t_plus_1"])
df = df.merge(assignment, on=KEY, how="left", validate="one_to_one")

train = df[df["split_company"] == "train"].copy()
test = df[df["split_company"] == "test"].copy()
print(f"train {len(train):,} rows / test {len(test):,} rows")

train 65,293 rows / test 16,310 rows


In [9]:
# Evaluation
def discrimination_rate(y_true, y_pred, n_pairs=N_PAIRS, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    i, j = rng.integers(0, n, n_pairs), rng.integers(0, n, n_pairs)
    answerable = (i != j) & (y_true[i] != y_true[j])
    if not answerable.any():
        return np.nan
    return float((y_pred[i[answerable]] != y_pred[j[answerable]]).mean())


def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if np.std(y_pred) < 1e-12:
        return {"discrim_rate": 0.0, "pairwise_acc": np.nan,
                "effective_acc": 0.5, "spearman": np.nan}
    
    tau = kendalltau(y_true, y_pred, variant="b").statistic
    acc = (tau + 1) / 2 if np.isfinite(tau) else np.nan
    disc = discrimination_rate(y_true, y_pred)
    return {
        "discrim_rate": disc, 
        "pairwise_acc": acc,
        "effective_acc": 0.5 + disc * (acc - 0.5) if np.isfinite(acc) else np.nan,
        "spearman": spearmanr(y_true, y_pred).statistic
    }

## 1. The final model

`fit_final` trains once on train and returns a model plus the fitted statistics; `predict_oof`
produces train's out-of-fold predictions, each from a model that has not seen that company.

`employees` uses a two-part structure. The rest use a single regressor, both sharing the same
feature construction and folds.

In [10]:
def eligible_subset(frame, target):
    """rows on which this field can be modelled"""
    elig = frame[f"{target}_change_eligible"].fillna(False).astype(bool)
    sub = frame.loc[elig]
    y = sub[f"{target}_signed_log_change"].astype(float)
    return sub.loc[y.notna()], y.loc[y.notna()]

def fit_final(frame, target):
    """Fit the final model on the given data"""
    sub, y = eligible_subset(frame, target)
    X, cat_cols, stats = dp.build_matrix(sub, "gbm")
    levels = {c: pd.Categorical(X[c]).categories for c in cat_cols}
    for c in cat_cols:
        X[c] = pd.Categorical(X[c], categories=levels[c])

    common = dict(stats=stats, cat_cols=cat_cols, levels=levels, columns=X.columns)
    if target == TWO_PART_FIELD:
        moved = (y != 0).to_numpy()
        return {"kind": "two_part", **common,
                "clf": LGBMClassifier(**FINAL_PARAMS).fit(X, moved),
                "reg": LGBMRegressor(objective="quantile", alpha=0.5,
                                     **FINAL_PARAMS).fit(X[moved], y[moved])}
    return {"kind": "single", **common,
            "reg": LGBMRegressor(objective="quantile", alpha=0.5,
                                 **FINAL_PARAMS).fit(X, y)}


def predict_with(fitted, frame, target):
    """Predict with a fitted model"""
    sub, y = eligible_subset(frame, target)
    X, _, _ = dp.build_matrix(sub, "gbm", fit_stats=fitted["stats"])
    X = X.reindex(columns=fitted["columns"])
    for c, lv in fitted["levels"].items():
        X[c] = pd.Categorical(X[c], categories=lv)

    out = pd.DataFrame(index=sub.index)
    out["y_true"] = y
    if fitted["kind"] == "two_part":
        p = fitted["clf"].predict_proba(X)[:, 1]
        s = fitted["reg"].predict(X)
        out["p_change"], out["pred_size"], out["pred"] = p, s, p * s
    else:
        out["pred"] = fitted["reg"].predict(X)
    return out


def predict_oof(frame, target):
    """out-of-fold predictions within train"""
    sub, y = eligible_subset(frame, target)
    parts = []
    for tr, te in GroupKFold(N_SPLITS).split(sub, y, groups=sub["CompanyNumber_norm"]):
        fitted = fit_final(sub.iloc[tr], target)
        parts.append(predict_with(fitted, sub.iloc[te], target))
    return pd.concat(parts).reindex(sub.index)

In [11]:
# Fit and predict on both sides
final_models, oof_preds, test_preds = {}, {}, {}
for target in dp.METRICS:
    final_models[target] = fit_final(train, target)
    oof_preds[target] = predict_oof(train, target)
    test_preds[target] = predict_with(final_models[target], test, target)
    print(f"  done: {target}")

  done: current_assets
  done: fixed_assets
  done: creditors_total
  done: equity
  done: net_assets_liabilities
  done: net_current_assets_liabilities
  done: cash
  done: debtors
  done: employees
  done: profit_loss
  done: total_assets_less_current_liabilities


## 2. Test set evaluation

In [12]:
rows = []
for target in dp.METRICS:
    p = test_preds[target]
    rows.append({"target": target, "n": len(p),
                 "zero_frac": float((p["y_true"] == 0).mean()),
                 **evaluate(p["y_true"], p["pred"])})

test_result = pd.DataFrame(rows).sort_values("effective_acc", ascending=False).round(4)
test_result.to_csv(OUT_DIR / "test_evaluation.csv", index=False, encoding="utf-8-sig")
print(test_result.to_string(index=False))

# the classifier for employees
emp = test_preds[TWO_PART_FIELD]
print(f"\nemployee classifier AUC: "
      f"{roc_auc_score(emp['y_true'] != 0, emp['p_change']):.4f}")

                               target     n  zero_frac  discrim_rate  pairwise_acc  effective_acc  spearman
                            employees 15246     0.7351        1.0000        0.6144         0.6144    0.2940
                          profit_loss   565     0.0177        1.0000        0.6129         0.6129    0.3280
                         fixed_assets  7989     0.2382        0.9360        0.6048         0.5981    0.2897
                                 cash  6202     0.0379        1.0000        0.5980         0.5980    0.2823
                      creditors_total 14779     0.0635        0.9999        0.5829         0.5829    0.2409
total_assets_less_current_liabilities 14087     0.0610        0.9995        0.5780         0.5779    0.2161
                              debtors  5198     0.1116        1.0000        0.5709         0.5709    0.2078
                       current_assets 13923     0.0468        0.9999        0.5658         0.5658    0.1899
       net_current_assets_li

## 3. The prediction table

**On duplicates** 

12.19% of rows come from companies contributing more than one pair. Using them all would give those
companies double weight in any percentile, and at deployment each company is scored once from its
latest period. Rather than dropping rows, the latest pair is flagged.

In [13]:
def build_wide(frame, preds, source_label):
    """The wide table for one side(train or test)"""
    base = frame[KEY + META_COLS].copy()
    base["pred_source"] = source_label

    for target in dp.METRICS:
        p = preds[target]
        base[f"{target}__x_t"] = frame[f"{target}_t"].to_numpy()
        for col, out_name in [("y_true", "y_true"), ("pred", "pred_L_quantile")]:
            base[f"{target}__{out_name}"] = p[col].reindex(frame.index)
        if target == TWO_PART_FIELD:
            for col in ("p_change", "pred_size"):
                base[f"{target}__{col}"] = p[col].reindex(frame.index)
    return base


wide = pd.concat([build_wide(train, oof_preds, "oof"),
                  build_wide(test, test_preds, "final_model")], ignore_index=True)

# flag the latest pair per company
wide["is_latest_pair"] = (
    wide.groupby("CompanyNumber_norm")["period_t_plus_1"].transform("max")
    == wide["period_t_plus_1"])

wide = wide.sort_values(KEY).reset_index(drop=True)

In [14]:
# Checks
pred_cols = [c for c in wide.columns if c.endswith("__pred_L_quantile")]
n_pred = wide[pred_cols].notna().sum(axis=1)

print(f"rows: {len(wide):,}")
print(f"companies: {wide['CompanyNumber_norm'].nunique():,}")
print(f"columns: {wide.shape[1]}")
print(f"is_latest_pair=True: {int(wide['is_latest_pair'].sum()):,}")
print(f"rows with no prediction: {int((n_pred == 0).sum()):,}")
print(f"\npred_source:\n{wide['pred_source'].value_counts().to_string()}")
print(f"\navailable fields per row:\n{n_pred.value_counts().sort_index().to_string()}")

wide.to_csv(OUT_DIR / "predictions_wide.csv", index=False, encoding="utf-8-sig")

rows: 81,603
companies: 76,611
columns: 43
is_latest_pair=True: 76,611
rows with no prediction: 189

pred_source:
pred_source
oof            65293
final_model    16310

available fields per row:
0       189
1       437
2      1093
3       685
4      2129
5      3874
6      8348
7     18270
8     26715
9     10932
10     8244
11      687


### Test set results

**Consistent with cross-validation. no sign of overfitting**

Eight of eleven fields score higher on test than in cross-validation, with a maximum of +0.0125(`cash`). All lie at or near the fold-to-fold standard deviation of 0.002–0.008 measured in `model_comparison.ipynb`.

The direction is also explicable: each cross-validated model sees 80% of train, whereas the test
predictions come from a model fitted on all of it.

**The two-part model(`employees`) holds up on test**

Effective accuracy of 0.6144 against 0.6139 in cross-validation, with a discrimination rate of 1.0000 and a classifier AUC of 0.7697. This field now ranks first on test, having been the worst of all fields under a single regressor. The reversal came from the change in model structure, not from tuning.

**`profit_loss` remains unusable**

Its scores rank near the top but rest on 565 rows. Its fold-to-fold variation on train was three to
five times that of other fields, and the sample here is smaller still. Not recommended for scoring.

**The only field(`fixed_assets`) below a 0.95 discrimination rate**

Its discrimination rate of 0.9360 reflects a 23.8% zero-change rate. The impact is limited.

---

## 4. Time-based extrapolation

The primary split is random by company, so train and test overlap in time. Deployment does not work
that way: the model is fitted on history and applied to periods not yet filed. `split_time` takes the latest 20% by filing date and excludes companies already seen earlier.

**A known limitation**

The data spans 24 months of monthly bulk files with 76.1% of periods falling in 2024. This is
therefore a single split rather than a rolling backtest, and the conclusion is correspondingly weaker.


Compared against the primary test result: similar figures indicate no drift.

In [15]:
# Fit on the earlier data, evaluate on the latest segment

time_train = df[df["split_time"] == "time_train"]
time_test = df[df["split_time"] == "time_test"]
print(f"time_train {len(time_train):,}  / time_test {len(time_test):,} \n")

rows = []
for target in dp.METRICS:
    fitted = fit_final(time_train, target)
    p = predict_with(fitted, time_test, target)
    if len(p) < 200:  # too few rows to evaluate
        continue
    rows.append({"target": target, "n": len(p),
                 "zero_frac": float((p["y_true"] == 0).mean()),
                 **evaluate(p["y_true"], p["pred"])})

time_result = pd.DataFrame(rows).round(4)

time_train 72,094  / time_test 9,509 



In [16]:
# Against the primary test result
test_result = pd.read_csv(OUT_DIR / "test_evaluation.csv")
comp = (time_result[["target", "n", "effective_acc"]]
        .rename(columns={"n": "n_time", "effective_acc": "time_acc"})
        .merge(test_result[["target", "n", "effective_acc"]]
               .rename(columns={"n": "n_test", "effective_acc": "test_acc"}),
               on="target", how="outer"))
comp["diff"] = (comp["time_acc"] - comp["test_acc"]).round(4)
comp = comp.sort_values("diff")

print(comp.round(4).to_string(index=False))
print(f"\nmedian difference: {comp['diff'].median():+.4f}")
print(f"fields lower on the time split: "
      f"{int((comp['diff'] < 0).sum())}/{comp['diff'].notna().sum()}")

                               target  n_time  time_acc  n_test  test_acc    diff
                               equity    9148    0.5509   15735    0.5614 -0.0105
                                 cash    3448    0.5881    6202    0.5980 -0.0099
total_assets_less_current_liabilities    8212    0.5686   14087    0.5779 -0.0093
               net_assets_liabilities    7997    0.5568   13295    0.5621 -0.0053
                         fixed_assets    4556    0.5959    7989    0.5981 -0.0022
                              debtors    2810    0.5690    5198    0.5709 -0.0019
       net_current_assets_liabilities    8799    0.5635   14928    0.5647 -0.0012
                      creditors_total    8604    0.5884   14779    0.5829  0.0055
                            employees    8868    0.6200   15246    0.6144  0.0056
                       current_assets    7981    0.5724   13923    0.5658  0.0066
                          profit_loss     226    0.6307     565    0.6129  0.0178

median differen

**No systematic drift**

The median difference is −0.0019, with seven fields slightly lower on the time split and four
slightly higher. The three largest shortfalls sit at or just above the fold-to-fold standard deviation of 0.002–0.008, while the largest gain comes from a field with only 226 rows( `profit_loss` +0.0178).


This is a single split, not a rolling backtest: the data spans 24 months with 76.1% of periods in
2024, leaving no room for multiple consecutive windows. The claim supported here is that no marked
drift is visible within the observable range, not that the model is stable going forward.

## 5. Distribution of the two prediction sources

In [17]:
# The downstream pools both sides to fix its reference distribution, 
# so their percentile cut-offs must be comparable.
rows = []
for target in dp.METRICS:
    col = f"{target}__pred_L_quantile"
    for label in ("oof", "final_model"):
        s = wide.loc[wide["pred_source"] == label, col].dropna()
        if len(s) < 200:
            continue
        rows.append({"target": target, "pred_source": label, "n": len(s),
                     "p50": s.quantile(0.50), "p80": s.quantile(0.80),
                     "p90": s.quantile(0.90), "sd": s.std()})

src = pd.DataFrame(rows)
piv = src.pivot(index="target", columns="pred_source", values=["p80", "p90", "sd"])
piv[("p80", "diff")] = (piv[("p80", "final_model")] - piv[("p80", "oof")]).round(4)
piv[("p90", "diff")] = (piv[("p90", "final_model")] - piv[("p90", "oof")]).round(4)

print("percentile cut-offs on each side:")
print(piv.round(4).to_string())
print(f"\nP80 median difference: {piv[('p80', 'diff')].median():+.4f}")
print(f"P90 median difference: {piv[('p90', 'diff')].median():+.4f}")

percentile cut-offs on each side:
                                              p80                 p90                  sd             p80     p90
pred_source                           final_model     oof final_model     oof final_model     oof    diff    diff
target                                                                                                           
cash                                       0.1620  0.1704      0.3430  0.3524      0.5631  0.4664 -0.0084 -0.0093
creditors_total                            0.1111  0.1116      0.2017  0.2074      0.4795  0.5121 -0.0005 -0.0057
current_assets                             0.0833  0.0827      0.1694  0.1691      0.3030  0.2703  0.0006  0.0003
debtors                                    0.0672  0.0654      0.1198  0.1187      0.5016  0.4165  0.0018  0.0011
employees                                  0.0632  0.0620      0.0913  0.0911      0.1844  0.1790  0.0012  0.0002
equity                                     0.0917  0.0

**The two sources give near-identical cut-offs**
The downstream can therefore fix its reference distribution on the pooled data rather than treating
the two sources separately.


## 6. Feature contribution

**B2c**
A model using only the field's own level at t as its single feature. The difference from the final
model is the increment contributed by the other features. At the feasibility stage this ranged from 0.016 to 0.128. We now have metadata, missingindicators and the merged category variable were added.

### 6.1 the 11 fields as only features

In [18]:
def b2c_score(frame, target):
    sub, y = eligible_subset(frame, target)
    X = pd.DataFrame({"x": dp.signed_log1p(sub[f"{target}_t"])}, index=sub.index)
    pred = np.full(len(sub), np.nan)
    for tr, te in GroupKFold(N_SPLITS).split(sub, y, groups=sub["CompanyNumber_norm"]):
        model = LGBMRegressor(objective="quantile", alpha=0.5, **FINAL_PARAMS)
        model.fit(X.iloc[tr], y.iloc[tr])
        pred[te] = model.predict(X.iloc[te])
    return evaluate(y, pred)["effective_acc"]


rows = []
for target in dp.METRICS:
    full = oof_preds[target]
    rows.append({"target": target,
                 "b2c": b2c_score(train, target),
                 "full": evaluate(full["y_true"], full["pred"])["effective_acc"]})

b2c = pd.DataFrame(rows)
b2c["gain"] = (b2c["full"] - b2c["b2c"]).round(4)
b2c = b2c.sort_values("gain", ascending=False).round(4)

print(b2c.to_string(index=False))
print(f"\nmedian gain: {b2c['gain'].median():+.4f}")

                               target    b2c   full   gain
                            employees 0.5007 0.6139 0.1131
                               equity 0.5043 0.5565 0.0523
               net_assets_liabilities 0.5076 0.5582 0.0506
total_assets_less_current_liabilities 0.5257 0.5711 0.0454
                              debtors 0.5221 0.5660 0.0439
                       current_assets 0.5272 0.5668 0.0395
       net_current_assets_liabilities 0.5285 0.5650 0.0365
                      creditors_total 0.5460 0.5819 0.0359
                                 cash 0.5533 0.5853 0.0320
                         fixed_assets 0.5786 0.6030 0.0244
                          profit_loss 0.5844 0.6000 0.0155

median gain: +0.0395


The median increment from all other features is **+0.0395**, ranging from 0.0155(`profit_loss`) to 0.1131(`employees`). Using only the field's own level at t gives 0.5007–0.5844; the full feature set raises this to 0.5565–0.6139.

The increment runs inverse to the univariate baseline: `employees`, whose own level is close to chance at 0.5007, gains most, while the two fields with the strongest univariate baselines gain least, which indicate the less a field's own level explains, the more the remaining features supply.

The contribution exceeds that of the feasibility stage, whose range was 0.016–0.128 with a coarser
feature set. The median of +0.0395 is well beyond the fold-to-fold standard deviation of 0.002–0.008.

### 6.2 feature importance

**Design**
Shares of total gain by group.

| Group | Contents |
|---|---|
| `own level` | The target field's own base-period value at t, e.g. `employees_sl` when predicting the change in `employees` |
| `balance-sheet scale` | Identity-linked measures of overall balance-sheet size (net assets, total assets less current liabilities), **excluding the target's own value** |
| `liquidity composition` | The composition of liquid assets (cash, debtors, current assets), **excluding the target's own value** |
| `standalone fields` | Standalone and operational measures (fixed assets, creditors, headcount, profit or loss) |
| `category` | Sector and filing type |
| `missing indicators` | The missingness flags |
| `metadata` | The remaining metadata features |

The groups are mutually exclusive: the target's own level is assigned first and does not re-enter its cluster, so each row sums to one.

In [20]:
IDENTITY = ["equity", "net_assets_liabilities",
            "total_assets_less_current_liabilities", "net_current_assets_liabilities"]
LIQUIDITY = ["current_assets", "cash", "debtors"]
STANDALONE = ["creditors_total", "fixed_assets", "employees", "profit_loss"]
CATEGORICAL = ["primary_sector", "acct_cat_model", "evidence_tier_t"]


def assign_group(col, target):
    if col.endswith("_missing"):
        return "missing indicators"
    if col == f"{target}_sl":
        return "own level"
    for group, fields in [("balance-sheet scale", IDENTITY),
                          ("liquidity composition", LIQUIDITY),
                          ("standalone fields", STANDALONE)]:
        if any(col == f"{f}_sl" for f in fields):
            return group
    if col in CATEGORICAL:
        return "category"
    return "metadata"


rows = []
for target in dp.METRICS:
    m = final_models[target]
    model = m["clf" if m["kind"] == "two_part" else "reg"]
    imp = pd.Series(model.booster_.feature_importance("gain"),
                    index=model.feature_name_)
    grouped = imp.groupby([assign_group(c, target) for c in imp.index]).sum()
    rows.append((grouped / grouped.sum()).rename(target))
# Extract the split gain of each feature in the tree model, 
# sum it by the semantic groups above and 
# divide it by the total Gain to get the relative contribution ratio (the sum of each row is 1.0).
share = pd.DataFrame(rows).fillna(0)
share.loc["mean"] = share.mean()

print("share of total gain per field\n")
print(share.round(4).to_string())

share of total gain per field

                                       balance-sheet scale  category  liquidity composition  metadata  missing indicators  own level  standalone fields
current_assets                                      0.1013    0.0266                 0.0302    0.2729                 0.0     0.2652             0.3038
fixed_assets                                        0.0764    0.2877                 0.1125    0.0598                 0.0     0.4073             0.0563
creditors_total                                     0.1118    0.0162                 0.1209    0.0994                 0.0     0.5750             0.0766
equity                                              0.0880    0.1223                 0.0361    0.0408                 0.0     0.5928             0.1200
net_assets_liabilities                              0.3827    0.0180                 0.0440    0.1691                 0.0     0.0624             0.3238
net_current_assets_liabilities                      0.095

**A field's own level is central, at 29.5% on average**

The model predicts each field's change from the eleven base-period values, so a field's own level
carrying substantial weight is expected.

**Balance-sheet scale averages 20.6%, and must be read together with `own level`**

The four identity-cluster fields substitute for one another and the split choice between them is
unstable. A low `own level` share therefore does not mean the field's own level is unimportant: the
model has used other columns from the same cluster instead.

To see the overall contribution of the cluster, the two columns should be added (the two are mutually exclusive). equity 0.681, TALCL 0.499, NCA 0.457, net_assets 0.445

**Standalone fields average 22.6%**

They contribute most when predicting `debtors` and `net_current_assets_liabilities`, whose changes
draw more on operational indicators than on their own levels.

**Category features average 5.9%, but reach 28.8% for `fixed_assets`**

Far above the other fields

**The missing indicators contribute nothing**

The share is 0.000 in all eleven models: no split ever used them. LightGBM handles NaN natively, so
the pattern is already carried by the source columns. They are harmless but useless.